# Spider Experiment 1: Qwen3-VL-2B on MolmoWeb
This notebook runs domain-disjoint data preparation, base-model evaluation, QLoRA SFT, post-SFT evaluation, ScreenSpot transfer evaluation, and failure reporting. Enable Internet and a GPU in Kaggle before starting.

In [ ]:
from pathlib import Path
REPO_ROOT = Path.cwd()
assert (REPO_ROOT / 'pyproject.toml').exists(), 'Open this notebook from the repository root.'

In [ ]:
# Install the current official Transformers/TRL Qwen3-VL training stack.
%pip install -q -e ".[train]"

## Optional: restore a previous Kaggle output
Attach the previous notebook version's output as a Kaggle input, set `PREVIOUS_RUN_ROOT`, and run this before preparation/training. Leave it as `None` on the first run.

In [ ]:
from spider.workflow import gpu_summary, restore_run
print(gpu_summary())
PREVIOUS_RUN_ROOT = None  # Example: '/kaggle/input/spider-exp1-run-1'
restore_run(PREVIOUS_RUN_ROOT, REPO_ROOT)

## 1. Prepare fixed data manifests
This streams only the selected public datasets, resizes screenshots, writes message-level manifests, and verifies domain isolation. Existing complete manifests are reused.

In [ ]:
from spider.prepare import prepare_all
prepare_all('configs/experiment1.yaml')

## 2. Measure the untouched base model
Predictions are appended and flushed individually. Re-run this cell to resume after a clean stop. Use `--limit 8` for the first smoke test, then remove it for the official baseline.

In [ ]:
from spider.evaluate import evaluate
_, baseline_metrics = evaluate('configs/experiment1.yaml', 'baseline', None, ['molmoweb', 'screenspot'])
baseline_metrics

## 3. QLoRA SFT
Run a bounded chunk, save the notebook version, restore its output next session, then run this cell again. `--additional-steps` automatically resumes and caps at the configured one-epoch target. For a smoke test, replace it with `--max-steps 2`.

In [ ]:
from spider.train import train
train('configs/experiment1.yaml', additional_steps=500)

## 4. Post-SFT evaluation and comparison
Run after training reaches its configured target. It uses the same manifests, prompts, quantization, and greedy decoding as the baseline.

In [ ]:
from spider.archive import archive_results
from spider.evaluate import evaluate
from spider.workflow import compare_run_outputs
adapter = 'outputs/experiment1/adapter/final'
_, sft_metrics = evaluate('configs/experiment1.yaml', 'sft', adapter, ['molmoweb', 'screenspot'])
comparison_path, comparison = compare_run_outputs('configs/experiment1.yaml')
print(comparison_path.read_text())
archive_results('configs/experiment1.yaml')